# DJ-TLED in Python

A NumPy port of [`DJTLED.cpp`](../src/DJTLED.cpp): an explicit-dynamics finite element solver for a neo-Hookean solid meshed with 4-node tetrahedra ("T4" elements).

**Algorithm.** Zhang, J. (2021). *A direct Jacobian total Lagrangian explicit dynamics finite element algorithm for real-time simulation of hyperelastic materials.* International Journal for Numerical Methods in Engineering, 122(20), 5744-5772. <https://doi.org/10.1002/nme.6772>

That citation is the one the C++ program prints. Equation numbers in the comments below ("Eq. (11)", "Table 3", ...) are copied from the comments in the C++ source and were not checked against the paper.

**What the program does**

1. Reads the mesh, material, boundary conditions and time settings from `NH_001_tension.txt`.
2. Precomputes constants for every tetrahedron (volume, mass, reference Jacobian, ...).
3. Steps through time. Each step computes the elastic forces of every tetrahedron, adds them up at the nodes, and moves the nodes with the central-difference rule.
4. Writes the results as VTK files that ParaView can open.
5. Compares the final displacements with the reference solution in `U_DJ_Reference_ten_001.vtk`.

The numbers quoted in the comments (array shapes, counts, example values) are those of `NH_001_tension.txt`. They change if `INPUT_FILE` points at another model.

**How this differs from the C++**

| C++ | This notebook |
|---|---|
| Loops over tetrahedra one at a time, split across OpenMP threads | NumPy array operations that handle all tetrahedra at once (a plain-Python loop over 6713 tetrahedra x 4000 steps took about 200 seconds when tried) |
| `float` (32-bit) | `float64` by default, see `DTYPE` |
| `Node` and `T4` objects | Plain arrays: one row per node or per tetrahedron |
| Gravity forces accumulated per tetrahedron while reading the file | Same numbers, computed in `init_bc` as nodal mass x g |
| Per-element force storage plus index tables to avoid thread races | One `np.bincount` call adds the element forces at the nodes |
| Writes into the working directory, Windows path `timesteps\step_...` | Writes into `output/`, with `/` in paths |
| Fixed output file names (`brain_everyu.vtk`, `undeformed.vtk`, `simulation.pvd`, `step_...vtk`) | Output file names start with the input file's name, and the final deformed mesh is called `..._u.vtk`: `NH_001_tension.txt` gives `NH_001_tension_u.vtk`, `NH_001_tension_undeformed.vtk`, ... |
| Prints a line for every exported step | Prints only the 10% progress marks |
| Loops forever if `</BC>` is missing from the input | Stops at the end of the file |
| Indexes past the end of an empty list for a material other than `NH` | Raises `ValueError` |
| No comparison with a reference | Section 10 compares the result with a reference VTK file |

**Before running:** this notebook needs NumPy, and Jupyter needs `ipykernel`. In a terminal, from this folder: `.venv/bin/python -m pip install -r requirements.txt`

In [1]:
import math
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np

## 1. Settings

In [2]:
# Input file with the mesh, material, boundary conditions and time settings.
# The C++ program takes this as its command-line argument (argv[1]).
INPUT_FILE = Path("../models/NH_001_tension.txt")
# Reference solution for that input: a VTK file with one displacement vector per node. Only section 10 reads it.
REFERENCE_FILE = Path("../reference/U_DJ_Reference_ten_001.vtk")
# Folder that receives every result file. The C++ writes into the current working directory instead.
OUTPUT_DIR = Path("output")
# Number type of every floating-point array. float64 = 8 bytes, about 16 significant digits.
# The C++ uses `float`, which is np.float32 (4 bytes, about 7 digits).
DTYPE = np.float64

## 2. Small matrix helpers

The C++ file defines its own 3x3 matrix routines. In NumPy most of them are a single operator, and they work on a whole stack of matrices at once: an array of shape `(6713, 3, 3)` holds one 3x3 matrix per tetrahedron, and `A @ B` multiplies matrix `e` of `A` with matrix `e` of `B` for every `e`.

| C++ | NumPy |
|---|---|
| `mat33x33(A, B, AB)` | `A @ B` |
| `mat33Tx33(A, B, AB)` | `A.swapaxes(-1, -2) @ B` (A transposed times B) |
| `mat33x33T(A, B, AB)` | `A @ B.swapaxes(-1, -2)` (A times B transposed) |
| `mat34x34T(A, B, AB)` | `A @ B.swapaxes(-1, -2)` |
| `mat33xScalar(A, b, Ab)` | `A * b` |
| `matDet33`, `matInv33` | `det33`, `inv33` below |

`det33` and `inv33` keep the explicit formulas of the C++ instead of calling `np.linalg`, so the arithmetic is the same.

In [3]:
# Determinant of 3x3 matrices, the same cofactor expansion as matDet33 in the C++.
# `A[..., i, j]` picks entry (i, j) of every matrix: for A of shape (6713, 3, 3) it has shape (6713,).
def det33(A):
    # det A = a00*(a11*a22 - a12*a21) - a10*(a01*a22 - a02*a21) + a20*(a01*a12 - a02*a11): expansion down column 0.
    # Example: for the diagonal matrix diag(2, 3, 4) only the first term is non-zero: 2 * (3*4 - 0*0) = 24.
    return (A[..., 0, 0] * (A[..., 1, 1] * A[..., 2, 2] - A[..., 1, 2] * A[..., 2, 1])
            - A[..., 1, 0] * (A[..., 0, 1] * A[..., 2, 2] - A[..., 0, 2] * A[..., 2, 1])
            + A[..., 2, 0] * (A[..., 0, 1] * A[..., 1, 2] - A[..., 0, 2] * A[..., 1, 1]))


# Inverse and determinant of 3x3 matrices (matInv33 in the C++): inverse = adjugate / determinant.
# Entry (i, j) of the adjugate is +/- the 2x2 determinant left after deleting row j and column i of A.
def inv33(A):
    # One determinant per matrix: shape (6713, 3, 3) -> (6713,).
    det = det33(A)
    # Uninitialised array with the same shape and number type as A; every entry is filled in below.
    inv = np.empty_like(A)
    # adj(0,0): delete row 0 and column 0, leaving [[a11, a12], [a21, a22]].
    inv[..., 0, 0] = A[..., 1, 1] * A[..., 2, 2] - A[..., 1, 2] * A[..., 2, 1]
    # adj(0,1): delete row 1 and column 0, with a minus sign: -(a01*a22 - a02*a21).
    inv[..., 0, 1] = A[..., 0, 2] * A[..., 2, 1] - A[..., 0, 1] * A[..., 2, 2]
    # adj(0,2): delete row 2 and column 0, leaving [[a01, a02], [a11, a12]].
    inv[..., 0, 2] = A[..., 0, 1] * A[..., 1, 2] - A[..., 0, 2] * A[..., 1, 1]
    # adj(1,0): delete row 0 and column 1, with a minus sign: -(a10*a22 - a12*a20).
    inv[..., 1, 0] = A[..., 1, 2] * A[..., 2, 0] - A[..., 1, 0] * A[..., 2, 2]
    # adj(1,1): delete row 1 and column 1, leaving [[a00, a02], [a20, a22]].
    inv[..., 1, 1] = A[..., 0, 0] * A[..., 2, 2] - A[..., 0, 2] * A[..., 2, 0]
    # adj(1,2): delete row 2 and column 1, with a minus sign: -(a00*a12 - a02*a10).
    inv[..., 1, 2] = A[..., 0, 2] * A[..., 1, 0] - A[..., 0, 0] * A[..., 1, 2]
    # adj(2,0): delete row 0 and column 2, leaving [[a10, a11], [a20, a21]].
    inv[..., 2, 0] = A[..., 1, 0] * A[..., 2, 1] - A[..., 1, 1] * A[..., 2, 0]
    # adj(2,1): delete row 1 and column 2, with a minus sign: -(a00*a21 - a01*a20).
    inv[..., 2, 1] = A[..., 0, 1] * A[..., 2, 0] - A[..., 0, 0] * A[..., 2, 1]
    # adj(2,2): delete row 2 and column 2, leaving [[a00, a01], [a10, a11]].
    inv[..., 2, 2] = A[..., 0, 0] * A[..., 1, 1] - A[..., 0, 1] * A[..., 1, 0]
    # Divide every adjugate by its own determinant. `det[..., None, None]` reshapes (6713,) to (6713, 1, 1)
    # so that broadcasting stretches each determinant over the 3x3 entries of its matrix.
    inv /= det[..., None, None]
    # Return both, because the callers need the determinant as well (volume, and volume ratio J).
    return inv, det

## 3. Reading the model

`NH_001_tension.txt` is a list of whitespace-separated tokens (39831 of them). Its layout, with the counts found in this file:

```
0   0.100000001  0.100000001  0.100000001     node id, x, y, z                  1440 lines
...
NH 6567 326210                                material type, Mu, Kappa
Density 1060
T4                                            element type
0   602  603  604  605                        element id, its 4 node ids        6713 lines
...
<Disp>                                        prescribed displacement ...
y 0.05                                        ... direction and final size ...
0 3 6 7 35 ...                                ... and the node ids it applies to   121 ids
<FixP>                                        fixed position: x (121 ids), then z (121 ids), then all (121 ids)
...
</BC>
Damping 5
TimeStep 0.00025
TotalTime 1
```

The C++ reads this with `fscanf_s`, which consumes one token at a time and stops a `while` loop as soon as the next token is not a number. `TokenReader` does the same thing on a Python list.

The mesh fills a cube with side 0.1: the coordinates run from 0 to 0.1 in x, y and z, and the element volumes add up to 0.001. The 121 nodes with `<Disp>` are the face y = 0.1; they are also fixed in x and z. The 121 nodes fixed in all directions are the face y = 0. So one face is clamped and the opposite face is pulled away from it by 0.05, a tension test that stretches the cube by 50%.

In [4]:
# A cursor over the whitespace-separated tokens of the input file; it plays the role of fscanf_s in the C++.
class TokenReader:
    # `text` is the whole file as one string.
    def __init__(self, text):
        # str.split() with no argument cuts at any run of spaces, tabs or newlines:
        # "0\t-0.05  0\n" -> ["0", "-0.05", "0"]. NH_001_tension.txt gives 39831 tokens.
        self.tokens = text.split()
        # Index of the next unread token; starts at the first one.
        self.pos = 0

    # True if another token exists and it consists of digits only ("602" yes; "0.100000001", "NH", "<Disp>" no).
    # This is how the C++ loops end: fscanf_s("%u") fails on the first token that is not an unsigned integer.
    def has_uint(self):
        # `and` evaluates left to right and stops early, so tokens[pos] is never read past the end of the list.
        return self.pos < len(self.tokens) and self.tokens[self.pos].isdigit()

    # Return the next token as a string and move the cursor one step forward.
    def next_token(self):
        # Read the token under the cursor.
        token = self.tokens[self.pos]
        # Advance, so the following call returns the token after it.
        self.pos += 1
        # Hand the string to the caller, who converts it with int() or float() as needed.
        return token

    # Read integers until a non-integer token comes up; used for the node-id lists of the boundary conditions.
    def read_uints(self):
        # Collects the integers found so far.
        values = []
        # Keep going while the next token is all digits; stops at e.g. "<FixP>" without consuming it.
        while self.has_uint():
            # Convert the text "602" to the number 602 and store it.
            values.append(int(self.next_token()))
        # For the <Disp> block of NH_001_tension.txt this list has 121 node ids.
        return values

### The model container

`Model` holds everything that stays constant during the simulation. It replaces the C++ classes `Node`, `T4` and `Model`: instead of 1440 `Node` objects there is one `(1440, 3)` array of coordinates, and instead of 6713 `T4` objects there is one array per quantity with 6713 rows.

In [5]:
# @dataclass writes __init__ for us from the field list below, so Model(fname=..., node_xyz=..., ...) just works.
@dataclass
class Model:
    # Name of the input file; used for printing and as the prefix of the output file names.
    fname: str
    # Node coordinates, shape (1440, 3): row i = (x, y, z) of node i in the undeformed mesh.
    node_xyz: np.ndarray
    # Connectivity, shape (6713, 4): row e = the 4 node numbers of tetrahedron e, counted from 0.
    tet_nodes: np.ndarray
    # Material name; only "NH" (neo-Hookean) is implemented.
    material_type: str
    # Material constants [Mu, Kappa] = [6567.0, 326210.0]: shear modulus and bulk modulus.
    material_vals: list
    # Mass density, 1060.0.
    rho: float
    # Element name from the file, "T4".
    ele_type: str
    # Id of the first node in the file (0 here). Ids are shifted by it so that internal numbering starts at 0.
    node_begin_index: int
    # Id of the first element in the file (0 here); only printed.
    ele_begin_index: int
    # Number of boundary-condition blocks in the file (4 here: one <Disp>, three <FixP>).
    num_BCs: int
    # Prescribed displacements, three arrays of equal length (121 here): entry k says that
    # node disp_node[k] moves along axis disp_axis[k] (0 = x, 1 = y, 2 = z) by disp_mag[k] at the end of the run.
    disp_node: np.ndarray
    # Axis index of each prescribed displacement (all 1, i.e. y, in NH_001_tension.txt).
    disp_axis: np.ndarray
    # Final size of each prescribed displacement (all 0.05 in NH_001_tension.txt).
    disp_mag: np.ndarray
    # Fixed positions, two arrays of equal length (605 here): node fixP_node[k] cannot move along axis fixP_axis[k].
    fixP_node: np.ndarray
    # Axis index of each fixed position (242 entries for x, 121 for y, 242 for z in NH_001_tension.txt).
    fixP_axis: np.ndarray
    # Gravity acceleration (gx, gy, gz), shape (3,). NH_001_tension.txt has no <Gravity> block, so it is (0, 0, 0).
    gravity: np.ndarray
    # Mass-proportional damping coefficient ("Damping" in the file), 5.0.
    alpha: float
    # Time-step size, 0.00025.
    dt: float
    # Simulated time span, 1.0.
    total_t: float
    # Number of time steps, ceil(total_t / dt) = 4000.
    num_steps: int
    # Number of degrees of freedom: 3 per node, 1440 * 3 = 4320.
    num_DOFs: int
    # Reference Jacobian of every tetrahedron, shape (6713, 3, 3): row r of J0[e] is the edge vector
    # from local node 0 to local node r+1 in the undeformed mesh.
    J0: np.ndarray
    # Determinant of J0, shape (6713,); equals 6 x the tetrahedron volume.
    detJ0: np.ndarray
    # Volume of every tetrahedron, shape (6713,).
    Vol: np.ndarray
    # Mass of every tetrahedron, shape (6713,).
    mass: np.ndarray
    # Six constants per tetrahedron, shape (6713, 6), that turn the deformed metric into the invariant I1 (Eq. (C.1)).
    m1: np.ndarray
    # Constant 3x3 matrix per tetrahedron, shape (6713, 3, 3), used in the force formula (Eq. (37), scaled by Mu/2).
    I1m: np.ndarray
    # The constant -2/3 * Mu/2 = -2189.0 that appears in the force formula.
    twothird_DwDI1cap: float
    # Flat degree-of-freedom number (node * 3 + axis) of every element force entry, shape (6713 * 4 * 3,) = (80556,).
    dof_index: np.ndarray

### Per-element constants

This is the constructor of the C++ class `T4`, done for all tetrahedra at once.

**Reference Jacobian.** Call the undeformed positions of the four nodes of a tetrahedron $X_0, X_1, X_2, X_3$. The rows of $J_0$ are the three edge vectors that start at node 0:

$$J_0 = \begin{bmatrix} X_1 - X_0 \\ X_2 - X_0 \\ X_3 - X_0 \end{bmatrix}, \qquad V = \frac{\det J_0}{6}$$

The code gets $J_0$ as `DHDR @ n_coords`. `DHDR` holds the derivatives of the four linear shape functions with respect to the element's local coordinates; multiplying by it subtracts node 0 from nodes 1, 2 and 3.

**The invariant $I_1$.** Later, each step builds the same matrix from the deformed positions, $J_t$ (rows $x_a - x_0$). The deformation gradient $F$ maps undeformed edges to deformed edges, which gives $F^T = J_0^{-1} J_t$. The first invariant of the deformation is

$$I_1 = \operatorname{tr}(F^T F) = \sum_{i,j} C^0_{ij}\, g_{ij}, \qquad C^0 = J_0^{-T} J_0^{-1}, \quad g = J_t J_t^T$$

$C^0$ never changes, so it is computed here once. The C++ stores it as six numbers `m1` (one per independent entry of the symmetric matrix $g$), so that $I_1$ = `m1[0]*g11 + m1[1]*g22 + m1[2]*g33 + m1[3]*g12 + m1[4]*g13 + m1[5]*g23`. The last three are doubled entries of $C^0$, because $g_{12}$ and $g_{21}$ are the same number and are counted once.

The relation between `m1` and $C^0$, and the formulas in the next sections, are a reading of the code, not quotations from the paper. They were checked with numbers from this mesh: `m1` equals $(C^0_{11}, C^0_{22}, C^0_{33}, 2C^0_{12}, 2C^0_{13}, 2C^0_{23})$ for every tetrahedron up to rounding (relative difference about 1e-16), and for every undeformed tetrahedron $I_1$ is 3 to within 1e-14.

In [6]:
# Derivatives of the 4 linear shape functions with respect to the 3 local coordinates, shape (3, 4).
# Row r is "node r+1 minus node 0": multiplying by it turns 4 node positions into 3 edge vectors.
DHDR = np.array([[-1, 1, 0, 0],
                 [-1, 0, 1, 0],
                 [-1, 0, 0, 1]], dtype=DTYPE)

# Six symmetric "selector" matrices, shape (6, 3, 3), in the order 11, 22, 33, 12, 13, 23.
# The first three have a single 1 on the diagonal; the last three have a pair of 1s off the diagonal.
VOIGT_M = np.array([[[1, 0, 0], [0, 0, 0], [0, 0, 0]],
                    [[0, 0, 0], [0, 1, 0], [0, 0, 0]],
                    [[0, 0, 0], [0, 0, 0], [0, 0, 1]],
                    [[0, 1, 0], [1, 0, 0], [0, 0, 0]],
                    [[0, 0, 1], [0, 0, 0], [1, 0, 0]],
                    [[0, 0, 0], [0, 0, 1], [0, 1, 0]]], dtype=DTYPE)


# Compute the constants of every tetrahedron (the body of the T4 constructor in the C++).
def precompute_elements(node_xyz, tet_nodes, rho, material_vals):
    # Index the (1440, 3) coordinate array with the (6713, 4) node numbers -> shape (6713, 4, 3):
    # n_coords[e, a] is the (x, y, z) of local node a of tetrahedron e. (The C++ stores this transposed, as 3x4.)
    n_coords = node_xyz[tet_nodes]
    # (3, 4) @ (6713, 4, 3) -> (6713, 3, 3): NumPy repeats DHDR for every tetrahedron.
    # Row 0 of J0[0] is node 603 minus node 602 = (-0.001669005, -0.005118512, -0.007882003).
    J0 = DHDR @ n_coords
    # Inverse (6713, 3, 3) and determinant (6713,) of every reference Jacobian. detJ0[0] = 8.2131e-07.
    invJ0, detJ0 = inv33(J0)
    # Volume of a tetrahedron = determinant of its three edge vectors / 6. Vol[0] = 1.36885e-07;
    # all 6713 volumes are positive and add up to 0.001, the volume of the 0.1 x 0.1 x 0.1 cube.
    Vol = detJ0 / 6.0
    # Mass = density * volume. mass[0] = 1060 * 1.36885e-07 = 0.000145098; the whole body weighs 1.06.
    mass = rho * Vol
    # Transpose of every inverse: swapaxes(1, 2) exchanges the row and column axes, shape stays (6713, 3, 3).
    invJ0T = invJ0.swapaxes(1, 2)
    # G_k = invJ0 @ M_k @ invJ0^T for the six selector matrices, Eqs. (B.1-6).
    # `[:, None]` inserts an axis: (6713, 1, 3, 3) @ (6, 3, 3) @ (6713, 1, 3, 3) broadcasts to (6713, 6, 3, 3).
    G = invJ0[:, None, :, :] @ VOIGT_M @ invJ0T[:, None, :, :]
    # Trace (sum of the diagonal) of each of the 6 matrices, Eq. (C.1): (6713, 6, 3, 3) -> (6713, 6).
    # m1[0] = (28215.7, 17120, 14736.1, -33474, 26431, -21653.1).
    m1 = np.trace(G, axis1=2, axis2=3)
    # Neo-Hookean energy per unit volume: w = Mu/2 * (I1cap - 3) + Kappa/2 * (J - 1)^2,
    # so its derivative with respect to I1cap is Mu / 2 = 6567 / 2 = 3283.5.
    DwDI1cap = material_vals[0] / 2.0
    # I1m = (invJ0^T @ invJ0) * 2 * Vol * Mu/2, Eq. (37) times Mu/2; shape (6713, 3, 3), symmetric.
    # `(2.0 * Vol)[:, None, None]` has shape (6713, 1, 1) so each volume scales its own 3x3 matrix.
    I1m = (invJ0T @ invJ0) * (2.0 * Vol)[:, None, None] * DwDI1cap
    # -2/3 * Mu/2 = -2189.0; multiplies J^(-2/3) * I1 in the force formula.
    twothird_DwDI1cap = -2.0 / 3.0 * DwDI1cap
    # Give all seven results back to read_model, which stores them in the Model.
    return J0, detJ0, Vol, mass, m1, I1m, twothird_DwDI1cap

### Parsing the file

`read_model` follows `readModel` in the C++ line by line: nodes, material, density, element type, elements, boundary conditions, then the three time settings.

In [7]:
# Axis letter in the file -> column index in the (num_nodes, 3) arrays.
AXIS_INDEX = {"x": 0, "y": 1, "z": 2}
# Which axes a <FixP> block locks: one axis, or all three for "all".
FIXP_AXES = {"x": [0], "y": [1], "z": [2], "all": [0, 1, 2]}


# Read the input file and return a fully prepared Model (readModel plus Model::postCreate in the C++).
def read_model(path):
    # Read the whole file into one string and wrap it in a token cursor.
    reader = TokenReader(Path(path).read_text())
    # Node ids as written in the file.
    node_ids = []
    # One [x, y, z] list per node.
    node_xyz = []
    # A node line starts with an integer id. Runs 1440 times, then stops at the token "NH".
    while reader.has_uint():
        # First token of the line: the node id.
        node_ids.append(int(reader.next_token()))
        # Next three tokens: the coordinates, converted from text to floats.
        node_xyz.append([float(reader.next_token()), float(reader.next_token()), float(reader.next_token())])
    # The id of the first node (0 in NH_001_tension.txt). Subtracting it later makes internal node numbers start at 0.
    node_begin_index = node_ids[0]
    # List of 1440 [x, y, z] lists -> array of shape (1440, 3).
    node_xyz = np.array(node_xyz, dtype=DTYPE)
    # Next token: the material name, "NH".
    material_type = reader.next_token()
    # The C++ only fills in material constants for "NH" and would then read past the end of an empty list.
    if material_type != "NH":
        # Stop with a clear message instead.
        raise ValueError(f"unsupported material type: {material_type}")
    # Two numbers follow "NH": Mu = 6567.0 (shear modulus) and Kappa = 326210.0 (bulk modulus).
    material_vals = [float(reader.next_token()), float(reader.next_token())]
    # Skip the label "Density"; like the C++, the label itself is not checked.
    reader.next_token()
    # The density value, 1060.0.
    rho = float(reader.next_token())
    # The element type, "T4".
    ele_type = reader.next_token()
    # Element ids as written in the file.
    ele_ids = []
    # One [n1, n2, n3, n4] list per tetrahedron.
    tet_nodes = []
    # An element line starts with an integer id. Runs 6713 times, then stops at the token "<Disp>".
    while reader.has_uint():
        # First token of the line: the element id.
        ele_ids.append(int(reader.next_token()))
        # Next four tokens: the node ids of the tetrahedron. `_` is a throwaway loop variable (4 repetitions).
        tet_nodes.append([int(reader.next_token()) for _ in range(4)])
    # The id of the first element (0 in NH_001_tension.txt), kept for printing.
    ele_begin_index = ele_ids[0]
    # (6713, 4) integer array; subtracting node_begin_index from every entry converts file ids to 0-based row numbers.
    tet_nodes = np.array(tet_nodes, dtype=np.int64) - node_begin_index
    # Three parallel lists describing the prescribed displacements (node, axis, size).
    disp_node, disp_axis, disp_mag = [], [], []
    # Two parallel lists describing the fixed positions (node, axis).
    fixP_node, fixP_axis = [], []
    # Gravity acceleration, starts as (0, 0, 0) and is only changed by <Gravity> blocks.
    gravity = np.zeros(3, dtype=DTYPE)
    # Counts the boundary-condition blocks, for printing.
    num_BCs = 0
    # Read block after block until "</BC>" (or until the tokens run out, which the C++ does not guard against).
    while reader.pos < len(reader.tokens):
        # The block header: "<Disp>", "<FixP>", "<Gravity>" or "</BC>".
        BC_type = reader.next_token()
        # Prescribed displacement block.
        if BC_type == "<Disp>":
            # Direction letter, "y" in NH_001_tension.txt.
            xyz = reader.next_token()
            # Displacement reached at the end of the run, 0.05 in NH_001_tension.txt.
            u = float(reader.next_token())
            # Only x, y and z are understood; anything else is skipped, as in the C++.
            if xyz in AXIS_INDEX:
                # The node ids this displacement applies to (121 of them in NH_001_tension.txt).
                for idx in reader.read_uints():
                    # Store the 0-based node number ...
                    disp_node.append(idx - node_begin_index)
                    # ... the axis as a column index (y -> 1) ...
                    disp_axis.append(AXIS_INDEX[xyz])
                    # ... and the size, one entry per node so the three lists stay the same length.
                    disp_mag.append(u)
            # One more boundary-condition block seen.
            num_BCs += 1
        # Fixed position block.
        elif BC_type == "<FixP>":
            # Direction: "x", "y", "z" or "all".
            xyz = reader.next_token()
            # Only the four known words are understood; anything else is skipped, as in the C++.
            if xyz in FIXP_AXES:
                # The node ids to fix (121 per block in NH_001_tension.txt).
                for idx in reader.read_uints():
                    # "all" yields three entries per node (axes 0, 1, 2), the others yield one.
                    for axis in FIXP_AXES[xyz]:
                        # Store the 0-based node number ...
                        fixP_node.append(idx - node_begin_index)
                        # ... and the axis it is locked along.
                        fixP_axis.append(axis)
            # One more boundary-condition block seen.
            num_BCs += 1
        # Gravity block (not present in NH_001_tension.txt).
        elif BC_type == "<Gravity>":
            # Direction letter.
            xyz = reader.next_token()
            # Acceleration along that direction, e.g. -9.81.
            g = float(reader.next_token())
            # Only x, y and z are understood.
            if xyz in AXIS_INDEX:
                # Add to that component; `+=` so repeated blocks accumulate, as they do in the C++.
                gravity[AXIS_INDEX[xyz]] += g
            # One more boundary-condition block seen.
            num_BCs += 1
        # End marker of the boundary-condition section.
        elif BC_type == "</BC>":
            # Leave the while loop; the time settings follow.
            break
    # Skip the label "Damping".
    reader.next_token()
    # Damping coefficient alpha, 5.0.
    alpha = float(reader.next_token())
    # Skip the label "TimeStep".
    reader.next_token()
    # Time-step size dt, 0.00025.
    dt = float(reader.next_token())
    # Skip the label "TotalTime".
    reader.next_token()
    # Simulated time span, 1.0.
    total_t = float(reader.next_token())
    # Steps needed to cover the time span, rounded up: ceil(1.0 / 0.00025) = 4000.
    num_steps = math.ceil(total_t / dt)
    # Each node can move in x, y and z: 1440 * 3 = 4320 degrees of freedom.
    num_DOFs = len(node_xyz) * 3
    # Constants of all tetrahedra; the tuple of seven results is unpacked into seven names.
    J0, detJ0, Vol, mass, m1, I1m, twothird_DwDI1cap = precompute_elements(node_xyz, tet_nodes, rho, material_vals)
    # Flat degree-of-freedom number of each element force entry: node * 3 + axis, the indexing the C++ uses.
    # (6713, 4, 1) * 3 + (3,) broadcasts to (6713, 4, 3); ravel() flattens it row by row to (80556,).
    # Example: node 602 owns degrees of freedom 1806 (x), 1807 (y) and 1808 (z).
    dof_index = (tet_nodes[:, :, None] * 3 + np.arange(3)).ravel()
    # Pack everything into a Model. The Python lists of boundary conditions become arrays here.
    return Model(
        fname=str(path), node_xyz=node_xyz, tet_nodes=tet_nodes,
        material_type=material_type, material_vals=material_vals, rho=rho, ele_type=ele_type,
        node_begin_index=node_begin_index, ele_begin_index=ele_begin_index, num_BCs=num_BCs,
        # Integer arrays so they can be used as row and column indices.
        disp_node=np.array(disp_node, dtype=np.int64), disp_axis=np.array(disp_axis, dtype=np.int64),
        disp_mag=np.array(disp_mag, dtype=DTYPE),
        fixP_node=np.array(fixP_node, dtype=np.int64), fixP_axis=np.array(fixP_axis, dtype=np.int64),
        gravity=gravity, alpha=alpha, dt=dt, total_t=total_t, num_steps=num_steps, num_DOFs=num_DOFs,
        J0=J0, detJ0=detJ0, Vol=Vol, mass=mass, m1=m1, I1m=I1m, twothird_DwDI1cap=twothird_DwDI1cap,
        dof_index=dof_index,
    )

In [8]:
# Print a summary of the model (printInfo in the C++). `:g` formats a float like C++ streams do: 0.00025 -> "0.00025".
def print_info(model):
    # Credit for the original implementation and the paper it implements.
    print("NumPy port of the open-source DJ-TLED implementation by Jinao Zhang:")
    # Title of the paper, first half.
    print("  Zhang, J. (2021). A direct Jacobian total Lagrangian explicit dynamics finite element algorithm")
    # Title of the paper, second half.
    print("  for real-time simulation of hyperelastic materials.")
    # Journal reference and DOI.
    print("  International Journal for Numerical Methods in Engineering, 122(20), 5744-5772. doi:10.1002/nme.6772")
    # Input file name: NH_001_tension.txt.
    print(f"Model:        {model.fname}")
    # len() of a 2-D array is its number of rows: 1440 nodes, 4320 degrees of freedom.
    print(f"Nodes:        {len(model.node_xyz)} ({model.num_DOFs} DOFs)")
    # 6713 elements of type T4.
    print(f"Elements:     {len(model.tet_nodes)} ({model.ele_type})")
    # Join the material constants with spaces: "6567 326210".
    material = " ".join(f"{val:g}" for val in model.material_vals)
    # Material name, its constants and the density.
    print(f"EleMaterial:  {model.material_type}: {material}; Density: {model.rho:g}")
    # Number of boundary-condition blocks: 4.
    print(f"BC:           {model.num_BCs}")
    # Damping coefficient: 5.
    print(f"DampingCoef.: {model.alpha:g}")
    # Time-step size: 0.00025.
    print(f"TimeStep:     {model.dt:g}")
    # Simulated time span: 1.
    print(f"TotalTime:    {model.total_t:g}")
    # Number of time steps: 4000.
    print(f"NumSteps:     {model.num_steps}")
    # First node id in the file: 0.
    print(f"Node index starts at {model.node_begin_index}.")
    # First element id in the file: 0.
    print(f"Elem index starts at {model.ele_begin_index}.")

## 4. Simulation state

`ModelStates` holds everything that changes during the run, plus the constants of the time integration. All nodal arrays have shape `(1440, 3)`: row = node, column = x, y or z. The C++ stores the same numbers in flat vectors of length 4320, indexed `node * 3 + axis`.

**Central-difference time integration.** For one degree of freedom with mass $m$, displacement $u$ and damping force $\alpha m \dot u$, Newton's second law is

$$m\,\ddot u + \alpha m\,\dot u = F_{ext} - F_{int}$$

Replace the derivatives by differences of the previous, current and next displacement, $\ddot u \approx (u^{+} - 2u + u^{-})/\Delta t^2$ and $\dot u \approx (u^{+} - u^{-})/(2\Delta t)$, and solve for $u^{+}$:

$$u^{+} = c_1\,(F_{ext} - F_{int}) + c_2\,u + c_3\,u^{-}$$

$$c_1 = \frac{1}{\dfrac{\alpha m}{2\Delta t} + \dfrac{m}{\Delta t^2}}, \qquad c_2 = \frac{2m}{\Delta t^2}\,c_1, \qquad c_3 = \frac{\alpha m}{2\Delta t}\,c_1 - \frac{c_2}{2}$$

Because every node has its own mass and no matrix couples the nodes ("lumped" mass), the update is a separate formula per degree of freedom; no linear system is solved.

In [9]:
# Everything that changes from step to step, plus the per-node integration constants (class ModelStates in the C++).
@dataclass
class ModelStates:
    # Mass of each node, shape (1440, 1). The trailing 1 lets it broadcast against (1440, 3) arrays.
    nodal_mass: np.ndarray
    # External force on each node, shape (1440, 3). Only gravity contributes, so it is all zeros for NH_001_tension.txt.
    external_F: np.ndarray
    # Prescribed displacement at the current time, shape (1440, 3); 0 means "not prescribed".
    disp_mag_t: np.ndarray
    # c1 of the central-difference formula, shape (1440, 1).
    central_diff_const1: np.ndarray
    # c2, shape (1440, 1).
    central_diff_const2: np.ndarray
    # c3, shape (1440, 1).
    central_diff_const3: np.ndarray
    # Displacements one step ago, shape (1440, 3).
    prev_U: np.ndarray
    # Displacements now, shape (1440, 3).
    curr_U: np.ndarray
    # True where a degree of freedom is fixed, shape (1440, 3).
    fixP_flag: np.ndarray


# Build the initial state: everything at rest, constants computed (the ModelStates constructor in the C++).
def create_model_states(model):
    # Number of rows of the coordinate array: 1440.
    num_nodes = len(model.node_xyz)
    # Start every node's mass at 0, shape (1440,).
    nodal_mass = np.zeros(num_nodes, dtype=DTYPE)
    # Give each tetrahedron's mass to its 4 nodes in equal quarters. np.add.at(target, index, values) does
    # target[index] += values and, unlike plain `+=`, counts a node once for every tetrahedron it belongs to
    # (between 4 and 40 in this mesh). `[:, None]` makes the values (6713, 1) to match the (6713, 4) index.
    np.add.at(nodal_mass, model.tet_nodes, (model.mass / 4.0)[:, None])
    # (1440,) -> (1440, 1). nodal_mass[0, 0] = 0.000114262; the masses add up to 1.06, the mass of the body.
    nodal_mass = nodal_mass[:, None]
    # c1 = 1 / (alpha*m/(2*dt) + m/dt^2). For node 0: 0.000546649.
    const1 = 1.0 / (model.alpha * nodal_mass / 2.0 / model.dt + nodal_mass / model.dt / model.dt)
    # c2 = 2*m*c1/dt^2. The mass cancels: c2 = 2 / (1 + alpha*dt/2) = 2 / 1.000625 = 1.99875078 for every node.
    const2 = 2.0 * nodal_mass * const1 / model.dt / model.dt
    # c3 = alpha*m*c1/(2*dt) - c2/2 = -0.99875078 for every node. Note c2 + c3 = 1.
    const3 = model.alpha * nodal_mass * const1 / 2.0 / model.dt - const2 / 2.0
    # Pack the state. Each np.zeros call makes its own new array, so the fields do not share memory.
    return ModelStates(
        nodal_mass=nodal_mass,
        external_F=np.zeros((num_nodes, 3), dtype=DTYPE),
        disp_mag_t=np.zeros((num_nodes, 3), dtype=DTYPE),
        central_diff_const1=const1, central_diff_const2=const2, central_diff_const3=const3,
        prev_U=np.zeros((num_nodes, 3), dtype=DTYPE),
        curr_U=np.zeros((num_nodes, 3), dtype=DTYPE),
        # dtype=bool: an array of False.
        fixP_flag=np.zeros((num_nodes, 3), dtype=bool),
    )

## 5. Boundary conditions

- **Gravity** is a constant external force, nodal mass x g, set once.
- **Fixed positions** are marked once in a True/False array.
- **Prescribed displacements** grow linearly in time, from 0 to their full size at the last step, so they are recomputed every step.

In [10]:
# Set the boundary conditions that do not change over time (initBC in the C++).
def init_bc(model, states):
    # Reset the external forces. `[:] = 0.0` overwrites the existing array in place instead of creating a new one.
    states.external_F[:] = 0.0
    # BC:Gravity. Force = mass * acceleration; (1440, 1) * (3,) broadcasts to (1440, 3). All zeros for NH_001_tension.txt.
    states.external_F += states.nodal_mass * model.gravity
    # BC:FixP. Indexing with two integer arrays pairs them up: entry k sets fixP_flag[fixP_node[k], fixP_axis[k]].
    # NH_001_tension.txt marks 605 of the 4320 degrees of freedom this way.
    states.fixP_flag[model.fixP_node, model.fixP_axis] = True


# Update the prescribed displacements for the step about to be computed (computeRunTimeBC in the C++).
def compute_run_time_bc(model, states, curr_step):
    # Fraction of the run completed after this step: step 0 -> 1 * 0.00025 / 1 = 0.00025; step 3999 -> 4000 * 0.00025 / 1 = 1.
    n = (curr_step + 1) * model.dt / model.total_t
    # BC:Disp. Scale each final displacement by that fraction and store it at its (node, axis) position.
    # Step 0 of NH_001_tension.txt: 0.05 * 0.00025 = 1.25e-05 on the y component of 121 nodes.
    states.disp_mag_t[model.disp_node, model.disp_axis] = model.disp_mag * n

## 6. One time step

This is `computeOneStep`, the core of the method.

**Element forces.** For every tetrahedron:

1. Deformed Jacobian (Eq. (11)): $J_t = J_0 + \begin{bmatrix} u_1 - u_0 \\ u_2 - u_0 \\ u_3 - u_0 \end{bmatrix}$, the edge vectors of the deformed tetrahedron.
2. Volume ratio (Eq. (12)): $J = \det J_t / \det J_0$. $J = 1$ means the volume is unchanged.
3. Invariant (Eq. (27)): $I_1 = \sum_k m_{1,k}\, g_k$ with $g = J_t J_t^T$.
4. Forces on local nodes 1, 2, 3 (rows of a 3x3 matrix; "Table 3 NH" in the C++ comments):

$$\begin{bmatrix} f_1 \\ f_2 \\ f_3 \end{bmatrix} = J^{-2/3}\; I_{1m}^T\, J_t \;+\; c\; J_t^{-T}, \qquad c = \left(-\tfrac{2}{3}\tfrac{\mu}{2}\, J^{-2/3} I_1 + \kappa\, J (J - 1)\right) V$$

5. Node 0 gets $f_0 = -(f_1 + f_2 + f_3)$, so the four forces of an element add up to zero.

This matrix is the derivative of the element's strain energy $V\,w$ with respect to $J_t$, for $w = \tfrac{\mu}{2}(J^{-2/3} I_1 - 3) + \tfrac{\kappa}{2}(J - 1)^2$. With no deformation ($J_t = J_0$) it gives $I_1 = 3$, $J = 1$ and zero force.

**Assembly.** A node belongs to several tetrahedra (4 to 40 in this mesh); its internal force is the sum of their contributions.

**Update.** The central-difference formula gives the next displacement of every free degree of freedom; fixed and prescribed ones are overwritten.

In [11]:
# Advance the displacements by one time step. Returns True on success, False if the solution blew up.
def compute_one_step(model, states):
    # Gather the current displacement of each tetrahedron's 4 nodes: (1440, 3) indexed by (6713, 4) -> (6713, 4, 3).
    u = states.curr_U[model.tet_nodes]
    # Eq. (11). u[:, 1:, :] is nodes 1..3, shape (6713, 3, 3); u[:, :1, :] is node 0 kept as shape (6713, 1, 3),
    # so the subtraction broadcasts node 0 against the other three. Result: deformed edge vectors, (6713, 3, 3).
    Jt = model.J0 + (u[:, 1:, :] - u[:, :1, :])
    # Eq. (17). Metric g = Jt @ Jt^T, shape (6713, 3, 3): g[e, i, j] is the dot product of deformed edges i and j.
    g = Jt @ Jt.swapaxes(1, 2)
    # The 6 independent entries of the symmetric g in the order 11, 22, 33, 12, 13, 23.
    # np.stack joins six (6713,) arrays along a new axis 1 -> (6713, 6), matching the layout of m1.
    g6 = np.stack([g[:, 0, 0], g[:, 1, 1], g[:, 2, 2], g[:, 0, 1], g[:, 0, 2], g[:, 1, 2]], axis=1)
    # Inverse (6713, 3, 3) and determinant (6713,) of the deformed Jacobians.
    invJt, detJt = inv33(Jt)
    # Eq. (12). Volume ratio, deformed volume / original volume, shape (6713,). Exactly 1 before anything moves.
    J = detJt / model.detJ0
    # J^(-2/3); the exponent is written as in the C++ (powf(J, -0.6666667f)). A negative J (an element turned
    # inside out) gives nan here, which the check at the end of this function catches.
    J23 = J ** -0.6666667
    # Eq. (27). I1 = sum of m1[k] * g6[k] over the 6 entries: (6713, 6) * (6713, 6), summed along axis 1 -> (6713,).
    # Equals 3 for an undeformed tetrahedron.
    I1 = (model.m1 * g6).sum(axis=1)
    # The scalar c in front of Jt^-T: (-2/3 * Mu/2 * J^(-2/3) * I1 + Kappa * J * (J - 1)) * Vol, shape (6713,).
    # The Kappa term is the pressure-like response to volume change; it vanishes when J = 1.
    const1 = (model.twothird_DwDI1cap * J23 * I1 + model.material_vals[1] * J * (J - 1.0)) * model.Vol
    # Forces on local nodes 1..3 (Table 3 NH), shape (6713, 3, 3): f123[e, a, c] = component c of the force on node a+1.
    # `[:, None, None]` turns the per-element scalars (6713,) into (6713, 1, 1) so they scale whole 3x3 matrices.
    # I1m is symmetric, so its transpose changes nothing mathematically; it is kept to match the C++ index order.
    f123 = J23[:, None, None] * (model.I1m.swapaxes(1, 2) @ Jt) + const1[:, None, None] * invJt.swapaxes(1, 2)
    # Force on local node 0 = minus the sum of the other three. keepdims=True keeps the summed axis: (6713, 1, 3).
    f0 = -f123.sum(axis=1, keepdims=True)
    # Put node 0 in front of nodes 1..3 along axis 1: (6713, 1, 3) + (6713, 3, 3) -> (6713, 4, 3).
    f = np.concatenate([f0, f123], axis=1)
    # Assembly. np.bincount(index, weights) adds up the weights that share the same index: here the 80556 element
    # force entries are summed into 4320 bins, one per degree of freedom. minlength keeps the output at 4320 even
    # if the last node had no entry. reshape(-1, 3) turns the flat (4320,) result back into (1440, 3).
    # bincount always returns float64, so astype converts back in case DTYPE is float32.
    internal_F = np.bincount(model.dof_index, weights=f.ravel(), minlength=model.num_DOFs).reshape(-1, 3).astype(DTYPE)
    # Explicit central-difference integration: u_next = c1 * (F_ext - F_int) + c2 * u_now + c3 * u_prev.
    # The (1440, 1) constants broadcast across the 3 columns of the (1440, 3) arrays.
    next_U = (states.central_diff_const1 * (states.external_F - internal_F)
              + states.central_diff_const2 * states.curr_U
              + states.central_diff_const3 * states.prev_U)
    # Apply BC:FixP. Indexing with a True/False array selects the True positions; those displacements are set to 0.
    next_U[states.fixP_flag] = 0.0
    # True where a displacement is prescribed. Like the C++, "prescribed" is detected as "non-zero", shape (1440, 3).
    has_disp = states.disp_mag_t != 0.0
    # Apply BC:Disp. Done after FixP so that it wins if a degree of freedom has both, as in the C++ if/else order.
    next_U[has_disp] = states.disp_mag_t[has_disp]
    # nan ("not a number") appears when the time step is too large and the solution diverges.
    # np.isnan marks nan entries, .any() is True if at least one exists, `not` flips it into "no error".
    no_err = not np.isnan(next_U).any()
    # Normal case: accept the step.
    if no_err:
        # Shift the history: what was "now" becomes "previous", the new values become "now".
        # The tuple on the right is built first, so both assignments use the old states.curr_U.
        states.prev_U, states.curr_U = states.curr_U, next_U
    # Diverged: leave the state untouched and report.
    else:
        # Same message as the C++.
        print("\n\tError: solution diverged, simulation aborted. Try a smaller time step.")
    # Tell the caller whether to continue.
    return no_err

## 7. Writing results

`export_vtk` writes legacy ASCII VTK files, the format the C++ writes:

- during the run, `output/timesteps/NH_001_tension_step_000000.vtk`, `NH_001_tension_step_000040.vtk`, ... with the deformed mesh at that step;
- at the end, `output/NH_001_tension_u.vtk` (final deformed mesh) and `output/NH_001_tension_undeformed.vtk` (original mesh). Both carry the final displacement vectors.

Every file name starts with the name of the input file without its extension (`NH_001_tension` for `NH_001_tension.txt`). The rest of each name is the one the C++ uses, except for the final deformed mesh, which the C++ calls `brain_everyu.vtk` and this notebook calls `..._u.vtk`.

In [12]:
# Write the mesh and displacements to VTK (exportVTK in the C++). With step=None it writes the two final files
# and returns ""; with a step number it writes one time-step file and returns its name for the PVD index.
def export_vtk(model, states, step=None):
    # Prefix of every output file: the input file name without folder and extension,
    # "NH_001_tension.txt" -> "NH_001_tension".
    # Results of different models can then share the output folder without overwriting each other.
    prefix = Path(model.fname).stem
    # Name of the file that holds the original mesh, "NH_001_tension_undeformed.vtk"; it is needed twice below.
    undeformed_vtk = f"{prefix}_undeformed.vtk"
    # Final export, called once after the simulation.
    if step is None:
        # Two files: the deformed mesh, "NH_001_tension_u.vtk", and the undeformed mesh.
        outputs = [f"{prefix}_u.vtk", undeformed_vtk]
        # Nothing to add to the PVD index for the final files.
        returned_filename = ""
        # Progress message.
        print("exporting final results...")
    # Export of one time step during the simulation.
    else:
        # `:06d` pads the step number with zeros to 6 digits so files sort in time order:
        # step 40 of NH_001_tension.txt -> "timesteps/NH_001_tension_step_000040.vtk".
        returned_filename = f"timesteps/{prefix}_step_{step:06d}.vtk"
        # Only this one file is written.
        outputs = [returned_filename]
    # 1440 nodes.
    num_nodes = len(model.node_xyz)
    # 6713 tetrahedra.
    num_tets = len(model.tet_nodes)
    # VTK cell rows are "4 n1 n2 n3 n4": the count of nodes followed by the node numbers.
    # np.full(6713, 4) is a column of 4s; column_stack glues it to the (6713, 4) connectivity -> (6713, 5).
    cells = np.column_stack([np.full(num_tets, 4), model.tet_nodes])
    # Write each requested file.
    for vtk in outputs:
        # Point coordinates: original positions for the undeformed file, otherwise original + current displacement.
        points = model.node_xyz if vtk == undeformed_vtk else model.node_xyz + states.curr_U
        # Open the file for writing as text; `with` closes it automatically, even if an error occurs.
        with open(OUTPUT_DIR / vtk, "w") as fout:
            # Line 1 of a legacy VTK file: the format identifier, copied from the C++.
            fout.write("# vtk DataFile Version 3.8\n")
            # Line 2: a free title; the C++ uses the file name.
            fout.write(f"{vtk}\n")
            # Line 3: the numbers below are plain text, not binary.
            fout.write("ASCII\n")
            # Line 4: a mesh made of arbitrary cells.
            fout.write("DATASET UNSTRUCTURED_GRID\n")
            # Header of the point list: 1440 points stored as floats.
            fout.write(f"POINTS {num_nodes} float\n")
            # One "x y z" line per node. "%g" prints 6 significant digits, like C++ streams do by default.
            np.savetxt(fout, points, fmt="%g")
            # Header of the cell list: 6713 cells and 6713 * 5 = 33565 integers in total.
            fout.write(f"CELLS {num_tets} {num_tets * (4 + 1)}\n")
            # One "4 n1 n2 n3 n4" line per tetrahedron, as integers.
            np.savetxt(fout, cells, fmt="%d")
            # Header of the cell-type list.
            fout.write(f"CELL_TYPES {num_tets}\n")
            # One line per cell with the code 10, which is VTK's number for a tetrahedron.
            np.savetxt(fout, np.full(num_tets, 10), fmt="%d")
            # What follows is data attached to the 1440 points.
            fout.write(f"POINT_DATA {num_nodes}\n")
            # A vector per point, named "displacement".
            fout.write("VECTORS displacement float\n")
            # One "ux uy uz" line per node.
            np.savetxt(fout, states.curr_U, fmt="%g")
        # Only the two final files are announced, to keep the output short.
        if step is None:
            # Name of the file just written.
            print(f"\t{vtk}")
    # After the final export, confirm.
    if step is None:
        # Same message as the C++.
        print("VTK saved.")
    # "" for the final export, the time-step file name otherwise.
    return returned_filename

## 8. The simulation loop

`run_simulation` repeats *update boundary conditions -> compute one step -> maybe export* for all 4000 steps. It also writes `output/NH_001_tension_simulation.pvd`, a small XML index that lists the time-step files with their times; opening it in ParaView loads the whole animation.

A snapshot is exported every `num_steps // 100` steps (every 40 steps here) plus the last step: steps 0, 40, ..., 3960 and 3999, which is 101 files.

In [13]:
# Run all time steps (runSimulation in the C++). Returns the final state, or None if the solution diverged.
def run_simulation(model):
    # Fresh state: zero displacements, integration constants computed.
    states = create_model_states(model)
    # Gravity forces and fixed-position flags.
    init_bc(model, states)
    # Last progress percentage that was printed.
    progress = 0
    # Start of the timing. perf_counter() is a high-resolution clock in seconds.
    start_t = time.perf_counter()
    # Progress message.
    print("computing...")
    # Create output/timesteps. parents=True also creates output/; exist_ok=True means no error on a re-run.
    (OUTPUT_DIR / "timesteps").mkdir(parents=True, exist_ok=True)
    # Header lines of the PVD file (ParaView's collection format for time series); entries are appended in the loop.
    pvd_lines = ['<?xml version="1.0"?>', '<VTKFile type="Collection" version="0.1">', "  <Collection>"]
    # Export about 100 snapshots: every 4000 // 100 = 40 steps. max(1, ...) guards runs shorter than 100 steps.
    export_interval = max(1, model.num_steps // 100)
    # Tell the user.
    print(f"Exporting every {export_interval} steps (total steps: {model.num_steps})")
    # Becomes False if a step diverges; defined here so it also exists when num_steps is 0.
    no_err = True
    # The simulation loop: step = 0, 1, ..., 3999.
    for step in range(model.num_steps):
        # Print at every full 10%. Written with integers to avoid rounding: at step 399, 400 * 100 >= 10 * 4000.
        if (step + 1) * 100 >= (progress + 10) * model.num_steps:
            # Remember that this mark has been printed.
            progress += 10
            # Show it, e.g. "(10%)".
            print(f"\t({progress}%)")
        # Grow the prescribed displacements to their value for this step.
        compute_run_time_bc(model, states, step)
        # Compute forces and the new displacements.
        no_err = compute_one_step(model, states)
        # Diverged?
        if not no_err:
            # Stop stepping; the PVD file is still closed properly below.
            break
        # Export on steps 0, 40, 80, ... (remainder of step / 40 is 0) and on the very last step.
        if step % export_interval == 0 or step == model.num_steps - 1:
            # Write the snapshot and get its file name.
            vtk_filename = export_vtk(model, states, step)
            # Add it to the index with its time, step * dt. Step 40 -> timestep="0.01".
            pvd_lines.append(f'    <DataSet timestep="{step * model.dt:g}" file="{vtk_filename}"/>')
    # Closing tags of the PVD file.
    pvd_lines += ["  </Collection>", "</VTKFile>"]
    # The index file gets the same model prefix as the VTK files: output/NH_001_tension_simulation.pvd.
    pvd_path = OUTPUT_DIR / f"{Path(model.fname).stem}_simulation.pvd"
    # Join the lines with newlines and write the file in one go.
    pvd_path.write_text("\n".join(pvd_lines) + "\n")
    # A diverged run has no final state.
    if not no_err:
        # The caller checks for None.
        return None
    # Where to find the animation.
    print(f"Created {pvd_path} - open this file to view the complete animation")
    # Seconds since start_t, converted to milliseconds as in the C++.
    elapsed_ms = (time.perf_counter() - start_t) * 1000.0
    # `:.0f` prints the number without decimals.
    print(f"\nComputation time:\t{elapsed_ms:.0f} ms")
    # The final displacements live in states.curr_U.
    return states

## 9. Run

The equivalent of `main` in the C++.

In [14]:
# Read NH_001_tension.txt and precompute the element constants.
model = read_model(INPUT_FILE)
# Show what was read.
print_info(model)
# Run the 4000 steps.
states = run_simulation(model)
# Only export final results if the run did not diverge.
if states is not None:
    # Writes output/NH_001_tension_u.vtk and output/NH_001_tension_undeformed.vtk.
    export_vtk(model, states)

NumPy port of the open-source DJ-TLED implementation by Jinao Zhang:
  Zhang, J. (2021). A direct Jacobian total Lagrangian explicit dynamics finite element algorithm
  for real-time simulation of hyperelastic materials.
  International Journal for Numerical Methods in Engineering, 122(20), 5744-5772. doi:10.1002/nme.6772
Model:        NH_001_tension.txt
Nodes:        1440 (4320 DOFs)
Elements:     6713 (T4)
EleMaterial:  NH: 6567 326210; Density: 1060
BC:           4
DampingCoef.: 5
TimeStep:     0.00025
TotalTime:    1
NumSteps:     4000
Node index starts at 0.
Elem index starts at 0.
computing...
Exporting every 40 steps (total steps: 4000)
	(10%)
	(20%)
	(30%)
	(40%)
	(50%)
	(60%)
	(70%)
	(80%)
	(90%)
	(100%)
Created output/NH_001_tension_simulation.pvd - open this file to view the complete animation

Computation time:	6831 ms
exporting final results...
	NH_001_tension_u.vtk
	NH_001_tension_undeformed.vtk
VTK saved.


## 10. Check the result against the reference

`U_DJ_Reference_ten_001.vtk` is a reference solution for `NH_001_tension.txt`. It is a legacy VTK file with the same 1440 nodes and 6713 tetrahedra in the same order as the input, the deformed node positions, and one displacement vector per node (the block after the line `VECTORS ...`, written with 6 significant digits).

The cell below reads those vectors and compares them, number by number, with the final displacements of this run, `states.curr_U`. It reports:

- the **largest difference** in any single displacement component, and where it occurs;
- the **RMS difference**, the typical size of a difference: the square root of the mean of the squared differences;
- both again **relative to the largest displacement**, 0.05;
- how many of the 4320 components differ by more than `TOLERANCE`, and a final verdict.

**What counts as a match.** Exact equality cannot be expected: this notebook computes with 64-bit numbers, the C++ with 32-bit numbers, and the reference file keeps 6 digits. Two measurements show how large that rounding is for this model:

| comparison | largest difference |
|---|---|
| a 32-bit build of `DJTLED.cpp` vs the reference | 8e-07 |
| this notebook vs a 64-bit build of `DJTLED.cpp` | 5e-08 |

So differences up to about 1e-06 are rounding. `TOLERANCE` is set ten times higher, at 1e-05, which is 0.02% of the largest displacement. How the reference file was produced is not recorded in it; the first row suggests 32-bit arithmetic, but that is an inference.

**Result of the last run:** the largest difference was 1.21e-06 (node 32, y component: 0.0388259 here, 0.0388247 in the reference), the RMS difference 1.69e-07, and no component exceeded the tolerance.

In [15]:
# Largest difference from the reference that still counts as a match: 1e-05 is 0.02% of the largest displacement, 0.05.
TOLERANCE = 1e-5

# Skip the check if the run diverged.
if states is not None:
    # Read the reference file as a list of lines. splitlines() cuts at line breaks and drops them,
    # including the Windows-style "\r\n" endings this file uses. The file has 16315 lines.
    ref_lines = REFERENCE_FILE.read_text().splitlines()
    # Index of the first data line after the header "VECTORS ...". enumerate() yields (index, line) pairs and
    # next() takes the first index whose line starts with "VECTORS": 14874. Adding 1 skips the header itself.
    vec_start = next(i for i, line in enumerate(ref_lines) if line.startswith("VECTORS")) + 1
    # Two lines before the data is "POINT_DATA 1440"; its second word is the number of vectors in the file.
    num_ref = int(ref_lines[vec_start - 2].split()[1])
    # Number of nodes in this run: 1440.
    num_nodes = len(model.node_xyz)
    # The comparison only makes sense if both have one vector per node.
    print(f"nodes: {num_nodes} in this run, {num_ref} in {REFERENCE_FILE}")
    # Take the 1440 data lines, split each "ux uy uz" line into 3 strings, and convert to numbers: shape (1440, 3).
    # The last line of the file, "0.000844792 0.0496157 -0.00281591", becomes the last row.
    ref_U = np.array([line.split() for line in ref_lines[vec_start:vec_start + num_ref]], dtype=DTYPE)
    # Difference per node and axis, shape (1440, 3). (Raises an error here if the node counts differ.)
    diff = states.curr_U - ref_U
    # Largest absolute difference among the 4320 components.
    max_diff = np.abs(diff).max()
    # argmax() gives the position of that largest value in the flattened array (0..4319);
    # np.unravel_index converts it back to (row, column) = (node, axis). Position 97 would be node 32, axis 1.
    worst_node, worst_axis = np.unravel_index(np.abs(diff).argmax(), diff.shape)
    # Root mean square: square every difference, average the 4320 squares, take the square root.
    rms_diff = np.sqrt((diff ** 2).mean())
    # Largest displacement component in the reference, used as the scale for relative differences: 0.05.
    max_u = np.abs(ref_U).max()
    # Count the components whose difference exceeds the tolerance: (np.abs(diff) > TOLERANCE) is an array of
    # True/False, and sum() counts the True entries.
    num_over = int((np.abs(diff) > TOLERANCE).sum())
    # Largest displacement in the reference.
    print(f"largest displacement in the reference: {max_u:g}")
    # Largest difference and its location; "xyz"[worst_axis] turns the axis number 0, 1, 2 into the letter x, y, z.
    print(f"largest difference: {max_diff:.3g} at node {worst_node}, {'xyz'[worst_axis]} component")
    # The two numbers behind that difference.
    print(f"    this run {states.curr_U[worst_node, worst_axis]:g}, reference {ref_U[worst_node, worst_axis]:g}")
    # Typical difference.
    print(f"RMS difference:     {rms_diff:.3g}")
    # The same two measures divided by the largest displacement; `:.2e` prints e.g. 2.42e-05.
    print(f"relative to the largest displacement: largest {max_diff / max_u:.2e}, RMS {rms_diff / max_u:.2e}")
    # How many components are outside the tolerance, out of diff.size = 1440 * 3 = 4320.
    print(f"components differing by more than {TOLERANCE:g}: {num_over} of {diff.size}")
    # Verdict: a match if no component is outside the tolerance.
    print("MATCH" if num_over == 0 else "MISMATCH", f"(tolerance {TOLERANCE:g})")

nodes: 1440 in this run, 1440 in U_DJ_Reference_ten_001.vtk
largest displacement in the reference: 0.05
largest difference: 1.21e-06 at node 32, y component
    this run 0.0388259, reference 0.0388247
RMS difference:     1.69e-07
relative to the largest displacement: largest 2.42e-05, RMS 3.37e-06
components differing by more than 1e-05: 0 of 4320
MATCH (tolerance 1e-05)
